# Part 2 - Simple command-line chatbot (Notebook)

This notebook reproduces `chatbot.ipynb`. It loads a Hugging Face conversational model (`Qwen/Qwen2.5-0.5B-Instruct`) and enters a loop where you type messages and the bot replies.

**Exit commands:** `quit`, `exit`, `bye`.


In [1]:
pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 2.2 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.1 MB 2.2 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.1 MB 2.4 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.1 MB 2.6 MB/s eta 0:00:01
   ------------------------------ --------- 3.1/4.1 MB 2.8 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.1 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 3.0 MB/s  0:00:01
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/123.0 MB 5.0 MB/s eta 0:00:25
    --------------------------------------- 2.1/123.0 MB 5.4 MB/s eta 0:00:23
   - -----------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\fredb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
pip install transformers accelerate torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\fredb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import torch  # PyTorch: used to run the neural network model
from transformers import AutoModelForCausalLM, AutoTokenizer
# Hugging Face Transformers: provides pretrained LLMs and tokenizers

# =========================
# MODEL CONFIGURATION
# =========================

# The LLM used (Qwen instruction-tuned model)
# It is a small, efficient conversational AI model
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Commands to exit the chatbot
EXIT_COMMANDS = {"quit", "exit", "bye"}

# =========================
# LOAD TOKENIZER AND MODEL
# =========================

# Tokenizer: converts text into tokens (numbers the model understands)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Model: pretrained transformer that generates text
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Stores conversation history in token form
chat_history_ids = None

step = 0  # tracks number of conversation turns

# =========================
# START CHATBOT
# =========================

print("Welcome to the Qwen chatbot ")
print("Type 'quit', 'exit', or 'bye' to stop the conversation.")

# =========================
# MAIN CONVERSATION LOOP
# =========================

while True:

    # Get user input
    user_input = input("You: ").strip()

    # =========================
    # EXIT CONDITION
    # =========================
    if user_input.lower() in EXIT_COMMANDS:
        print("Bot: Goodbye! Thanks for chatting.")
        break

    # Handle empty input
    if not user_input:
        print("Bot: Please type a message so I can respond.")
        continue

    # =========================
    # TOKENIZATION STEP
    # =========================

    # Convert user input text into token IDs
    # Add EOS token to mark end of sentence
    new_user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    # =========================
    # HANDLE CONVERSATION HISTORY
    # =========================

    # If previous conversation exists, append new input
    if chat_history_ids is not None:
        bot_input_ids = torch.cat(
            [chat_history_ids, new_user_input_ids],
            dim=-1
        )
    else:
        bot_input_ids = new_user_input_ids

    # =========================
    # TEXT GENERATION (LLM INFERENCE)
    # =========================

    # The model predicts the next tokens based on input context
    chat_history_ids = model.generate(
        bot_input_ids,

        # maximum sequence length (input + output)
        max_length=1000,

        # padding token (required for some models)
        pad_token_id=tokenizer.eos_token_id,

        # enables probabilistic generation (more natural responses)
        do_sample=True,

        # nucleus sampling (keeps most likely tokens)
        top_p=0.92,

        # top-k sampling (limits vocabulary choices)
        top_k=50,

        # controls creativity (higher = more random)
        temperature=0.75,
    )

    # =========================
    # EXTRACT MODEL RESPONSE
    # =========================

    # Remove input tokens to keep only generated response
    response_ids = chat_history_ids[:, bot_input_ids.shape[-1]:]

    # Convert tokens back to readable text
    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True
    )

    # =========================
    # HANDLE EMPTY RESPONSE
    # =========================

    if not response.strip():
        response = "I am thinking about that. Could you tell me more?"

    # Print chatbot response
    print(f"Bot: {response}")

    # Increment conversation step counter
    step += 1

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4355.07it/s]


Welcome to the Qwen chatbot 🤖
Type 'quit', 'exit', or 'bye' to stop the conversation.
Bot: 
The study field of Bill Gates was computer science. He was a well-known inventor, entrepreneur, and philanthropist who founded Microsoft Corporation in 1975 with his father Paul Allen. While he had significant contributions to various fields such as technology and business, his primary focus throughout his career was on advancing information technology and networking technologies. His work at Microsoft encompassed not only software development but also the integration of hardware, network infrastructure, and artificial intelligence into personal computers. As a result, he played a pivotal role in shaping how people interacted with information and communication technologies, which are central to modern computing practices.
Bot: 
Yes, Bill Gates is still alive today. He passed away on April 28, 2023, at the age of 84. The cause of his death was unspecified, though it's known that he suffered from 